# Run 03 — Cadence Experiment

55 seeds, 50 generations, pop 30. Qwen3.5-9B. `max_logprob_gap=1.0`, `n_categories=50`.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from collections import Counter
from pymoo.indicators.hv import HV

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.size": 9, "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "text.color": "black", "axes.labelcolor": "black",
    "xtick.color": "black", "ytick.color": "black",
})

RUN = Path("../runs/03_cadence")
seed_dirs = sorted(RUN.glob("vlm_boundary_seed_*"))
N = len(seed_dirs)

OBJ_COLS = ["fitness_MatrixDistance_fro", "fitness_TextDist", "fitness_TgtBal"]
OBJ_SHORT = ["MatDist", "TextDist", "TgtBal"]

# ── Load everything ──
traces, convs, stats_list, contexts, pareto_data = [], [], [], [], []
for sd in seed_dirs:
    traces.append(pd.read_parquet(sd / "trace.parquet"))
    convs.append(pd.read_parquet(sd / "convergence.parquet"))
    stats_list.append(json.loads((sd / "stats.json").read_text()))
    ctx_path = sd / "context.json"
    contexts.append(json.loads(ctx_path.read_text()) if ctx_path.exists() else None)
    pfiles = sorted(sd.glob("pareto_*.json"), key=lambda p: int(p.stem.split("_")[1]))
    pareto_data.append([json.loads(f.read_text()) for f in pfiles])

# ── Derived tables ──
seed_df = pd.DataFrame([{
    "seed": i, "class_a": s["class_a"], "class_b": s["class_b"],
    "pair": f"{s['class_a']} vs {s['class_b']}",
    "initial_tgtbal": convs[i]["pareto_min_TgtBal"].iloc[0],
    "final_tgtbal": convs[i]["pareto_min_TgtBal"].iloc[-1],
    "n_pareto": s["n_pareto"], "runtime": s["runtime_seconds"],
    "image_dim": s.get("image_dim", len(contexts[i]["image_patch_positions"])),
} for i, s in enumerate(stats_list)])
seed_df["reduction_pct"] = (1 - seed_df["final_tgtbal"] / seed_df["initial_tgtbal"]) * 100
seed_df["converged"] = seed_df["final_tgtbal"] < 0.01

# Pareto table
rows = []
for si, pds in enumerate(pareto_data):
    img_dim = seed_df.loc[si, "image_dim"]
    for p in pds:
        g = np.array(p["genotype"])
        rows.append({
            "seed": si, "class_a": stats_list[si]["class_a"],
            "class_b": stats_list[si]["class_b"],
            "MatDist": p["fitness"][0], "TextDist": p["fitness"][1], "TgtBal": p["fitness"][2],
            "text": p["text"],
            "img_active": (g[:img_dim] != 0).sum(), "txt_active": (g[img_dim:] != 0).sum(),
            "sparsity": (g == 0).mean(),
        })
pareto_df = pd.DataFrame(rows)

print(f"Seeds: {N} | Evaluations: {N*50*30:,} | Pareto solutions: {len(pareto_df):,}")
print(f"Converged (TgtBal<0.01): {seed_df['converged'].sum()}/{N} ({seed_df['converged'].mean()*100:.0f}%)")
print(f"Runtime: {seed_df['runtime'].sum()/3600:.1f}h")

## 1 — Boundary Convergence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# ── Left: All seeds, log scale ──
ax = axes[0]
for i, conv in enumerate(convs):
    c = "#d0d0d0" if seed_df.loc[i, "converged"] else "#E0607E"
    ax.plot(conv["generation"], conv["pareto_min_TgtBal"], color=c, lw=0.6, alpha=0.7)

# Highlight stuck seeds
for i, conv in enumerate(convs):
    if not seed_df.loc[i, "converged"]:
        ax.plot(conv["generation"], conv["pareto_min_TgtBal"], color="#E0607E", lw=1.2,
                label=seed_df.loc[i, "pair"] if i == seed_df[~seed_df["converged"]].index[0] else None)

ax.set_yscale("log")
ax.set_xlabel("Generation")
ax.set_ylabel("min TgtBal (log)")
ax.set_title("All 55 seeds", fontweight="bold")
ax.axhline(0.01, color="black", ls=":", lw=0.8, alpha=0.5)
ax.text(48, 0.012, "0.01", fontsize=7, ha="right", va="bottom")

# ── Center: Aggregated — converged vs stuck ──
ax = axes[1]
conv_arr = np.array([conv["pareto_min_TgtBal"].values for conv in convs])
mask_ok = seed_df["converged"].values
mask_stuck = ~mask_ok
gens = convs[0]["generation"].values

for mask, color, label in [(mask_ok, "#2176AE", f"Converged (n={mask_ok.sum()})"),
                            (mask_stuck, "#E0607E", f"Stuck (n={mask_stuck.sum()})")]:
    if mask.sum() == 0:
        continue
    subset = conv_arr[mask]
    mu = subset.mean(axis=0)
    ci = 1.96 * subset.std(axis=0) / np.sqrt(mask.sum())
    ax.fill_between(gens, mu - ci, mu + ci, alpha=0.15, color=color)
    ax.plot(gens, mu, color=color, lw=2, label=label)

ax.set_yscale("log")
ax.set_xlabel("Generation")
ax.set_ylabel("min TgtBal (log)")
ax.set_title("Converged vs stuck seeds (mean +/- 95% CI)", fontweight="bold")
ax.legend(fontsize=7)

# ── Right: Convergence speed histogram ──
ax = axes[2]
gen_conv = []
for i, conv in enumerate(convs):
    below = conv[conv["pareto_min_TgtBal"] < 0.01]
    gen_conv.append(below["generation"].iloc[0] if len(below) > 0 else 51)

ax.hist(gen_conv, bins=np.arange(0, 53, 2), color="#2176AE", edgecolor="white", alpha=0.8)
ax.axvline(np.median([g for g in gen_conv if g < 51]), color="black", ls="--", lw=1.5,
           label=f"Median: gen {np.median([g for g in gen_conv if g < 51]):.0f}")
ax.set_xlabel("Generation first reaching TgtBal < 0.01")
ax.set_ylabel("Count")
ax.set_title("Convergence speed distribution", fontweight="bold")
ax.legend(fontsize=7)

fig.tight_layout()
plt.show()

## 2 — Multi-Objective Convergence

In [ ]:
# Aggregated convergence: Pareto-front min per objective, converged seeds only
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
gens = convs[0]["generation"].values
conv_idx = seed_df[seed_df["converged"]].index.values

obj_conv_keys = ["pareto_min_MatrixDistance_fro", "pareto_min_TextDist", "pareto_min_TgtBal"]

for ax, key, name in zip(axes, obj_conv_keys, OBJ_SHORT):
    arr = np.array([convs[i][key].values for i in conv_idx])
    mu = arr.mean(axis=0)
    q25, q75 = np.percentile(arr, [25, 75], axis=0)
    lo, hi = np.percentile(arr, [5, 95], axis=0)

    ax.fill_between(gens, lo, hi, alpha=0.08, color="#2176AE")
    ax.fill_between(gens, q25, q75, alpha=0.2, color="#2176AE")
    ax.plot(gens, mu, color="#2176AE", lw=2, label="Mean")
    ax.plot(gens, np.median(arr, axis=0), color="#2176AE", lw=1, ls="--", label="Median")
    ax.set_xlabel("Generation")
    ax.set_ylabel(name)
    ax.set_title(f"Pareto min {name} (n={len(conv_idx)})", fontweight="bold")
    ax.legend(fontsize=7)
    if name == "TgtBal":
        ax.set_yscale("log")

fig.tight_layout()
plt.show()

## 3 — Hypervolume Indicator

In [ ]:
# Compute HV per generation for a representative subset (converged seeds)
# Use a uniform reference point across all seeds
all_fit = np.vstack([df[OBJ_COLS].values for df in traces])
ref = all_fit.max(axis=0) * 1.1
hv_ind = HV(ref_point=ref)

# Sample up to 20 converged seeds for speed
sample_idx = conv_idx[:20]
hv_arr = np.zeros((len(sample_idx), 50))

for row, si in enumerate(sample_idx):
    df = traces[si]
    archive = np.empty((0, 3))
    for gen in range(50):
        gf = df[df["generation"] == gen][OBJ_COLS].values
        archive = np.vstack([archive, gf])
        # Fast non-dominated filtering
        dominated = np.zeros(len(archive), dtype=bool)
        for i in range(len(archive)):
            if dominated[i]:
                continue
            for j in range(len(archive)):
                if i != j and not dominated[j] and np.all(archive[j] <= archive[i]) and np.any(archive[j] < archive[i]):
                    dominated[i] = True
                    break
        archive = archive[~dominated]
        hv_arr[row, gen] = hv_ind(archive)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
gens = np.arange(50)

# Individual curves
for row in range(len(sample_idx)):
    ax1.plot(gens, hv_arr[row], color="#d0d0d0", lw=0.5, alpha=0.7)
mu = hv_arr.mean(axis=0)
ax1.plot(gens, mu, color="#2176AE", lw=2.5)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Hypervolume")
ax1.set_title(f"Per-seed HV (n={len(sample_idx)})", fontweight="bold")

# Normalized to gen-0 baseline
hv_norm = hv_arr / hv_arr[:, 0:1]
mu_n = hv_norm.mean(axis=0)
ci = 1.96 * hv_norm.std(axis=0) / np.sqrt(len(sample_idx))
ax2.fill_between(gens, mu_n - ci, mu_n + ci, alpha=0.2, color="#2176AE")
ax2.plot(gens, mu_n, color="#2176AE", lw=2.5)
ax2.set_xlabel("Generation")
ax2.set_ylabel("HV / HV(gen 0)")
ax2.set_title("Relative HV gain (mean +/- 95% CI)", fontweight="bold")

fig.tight_layout()
plt.show()
print(f"Mean HV gain gen 0→49: {(mu_n[-1]-1)*100:.1f}%")

## 4 — Decision Boundary Landscape

Each point is one individual in (P(A), P(B)) space, colored by generation. Diagonal = boundary.

In [ ]:
# Show 6 representative seeds: 3 converged (fast, medium, slow) + 3 stuck
fast_conv = seed_df[seed_df["converged"]].nsmallest(1, "final_tgtbal").index[0]
mid_conv = seed_df[seed_df["converged"]].iloc[len(conv_idx)//2].name
slow_conv = seed_df[seed_df["converged"]].nlargest(1, "final_tgtbal").index[0]
stuck_seeds = seed_df[~seed_df["converged"]].nlargest(3, "final_tgtbal").index.tolist()
show = [fast_conv, mid_conv, slow_conv] + stuck_seeds[:3]
show = show[:6]  # cap at 6

fig, axes = plt.subplots(1, len(show), figsize=(3.5 * len(show), 3.5))
for col, si in enumerate(show):
    ax = axes[col]
    df = traces[si]
    sc = ax.scatter(df["p_class_a"], df["p_class_b"], c=df["generation"],
                    cmap="viridis", s=4, alpha=0.4, rasterized=True)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("P(A)"); ax.set_ylabel("P(B)")
    tb = seed_df.loc[si, "final_tgtbal"]
    tag = "conv" if seed_df.loc[si, "converged"] else "stuck"
    ax.set_title(f"S{si} [{tag}] TB={tb:.4f}\n{stats_list[si]['class_a'][:10]} v {stats_list[si]['class_b'][:10]}",
                fontsize=7, fontweight="bold")

fig.colorbar(sc, ax=axes.tolist(), label="Generation", shrink=0.8, pad=0.02)
fig.tight_layout()
plt.show()

## 5 — Per-Class-A Convergence

Seeds grouped by their source class. Identifies systematically hard/easy categories.

In [ ]:
# Group by class_a, show final TgtBal distribution
class_groups = seed_df.groupby("class_a")
classes = sorted(class_groups.groups.keys(), key=lambda c: class_groups.get_group(c)["final_tgtbal"].median())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

# Box plot of final TgtBal by class_a
data = [class_groups.get_group(c)["final_tgtbal"].values for c in classes]
bp = ax1.boxplot(data, labels=[c[:15] for c in classes], patch_artist=True, vert=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#2176AE")
    patch.set_alpha(0.6)
ax1.set_ylabel("Final TgtBal")
ax1.set_title("Final TgtBal by source class", fontweight="bold")
ax1.tick_params(axis="x", rotation=45)
ax1.set_yscale("log")
ax1.axhline(0.01, color="black", ls=":", lw=0.8)

# Convergence rate by class
conv_rate = class_groups["converged"].mean()
counts = class_groups.size()
x = np.arange(len(classes))
bars = ax2.bar(x, [conv_rate[c] for c in classes], color="#2176AE", alpha=0.8, edgecolor="white")
for i, c in enumerate(classes):
    ax2.text(i, conv_rate[c] + 0.02, f"{counts[c]}", ha="center", fontsize=7, color="gray")
ax2.set_xticks(x)
ax2.set_xticklabels([c[:15] for c in classes], rotation=45, ha="right")
ax2.set_ylabel("Convergence rate")
ax2.set_title("Fraction converged (TgtBal<0.01) per class — numbers = seed count", fontweight="bold")
ax2.set_ylim(0, 1.15)
ax2.axhline(1.0, color="gray", ls=":", lw=0.5)

fig.tight_layout()
plt.show()

## 6 — Pareto Front Projections

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

pairs = [("TgtBal", "MatDist"), ("TgtBal", "TextDist"), ("MatDist", "TextDist")]
titles = ["Image cost to reach boundary", "Text cost to reach boundary", "Modality trade-off"]

for ax, (xk, yk), title in zip(axes, pairs, titles):
    conv_mask = pareto_df["seed"].isin(conv_idx)
    stuck_mask = ~conv_mask

    if stuck_mask.any():
        ax.scatter(pareto_df.loc[stuck_mask, xk], pareto_df.loc[stuck_mask, yk],
                  s=12, alpha=0.3, color="#E0607E", label="Stuck seeds", rasterized=True)
    ax.scatter(pareto_df.loc[conv_mask, xk], pareto_df.loc[conv_mask, yk],
              s=12, alpha=0.3, color="#2176AE", label="Converged seeds", rasterized=True)
    ax.set_xlabel(xk)
    ax.set_ylabel(yk)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7, markerscale=2)

fig.tight_layout()
plt.show()

## 7 — Perturbation Budget vs Boundary Distance

Total perturbation cost (image + text) needed to reach the boundary. Each point is one Pareto solution.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

conv_p = pareto_df[pareto_df["seed"].isin(conv_idx)].copy()
conv_p["total_perturb"] = conv_p["MatDist"] + conv_p["TextDist"]

# Scatter: total perturbation vs TgtBal
sc = ax1.scatter(conv_p["total_perturb"], conv_p["TgtBal"],
                c=conv_p["MatDist"] / conv_p["total_perturb"],
                cmap="coolwarm", s=10, alpha=0.5, rasterized=True)
ax1.set_xlabel("Total perturbation (MatDist + TextDist)")
ax1.set_ylabel("TgtBal")
ax1.set_title("Perturbation budget vs boundary distance", fontweight="bold")
fig.colorbar(sc, ax=ax1, label="Image fraction of total", shrink=0.8)
ax1.axhline(0.01, color="black", ls=":", lw=0.8)

# Histogram of minimum-TgtBal pareto solution's total perturbation per seed
best_per_seed = conv_p.loc[conv_p.groupby("seed")["TgtBal"].idxmin()]
ax2.hist(best_per_seed["total_perturb"], bins=25, color="#2176AE", alpha=0.8, edgecolor="white")
ax2.axvline(best_per_seed["total_perturb"].median(), color="black", ls="--", lw=1.5,
           label=f"Median: {best_per_seed['total_perturb'].median():.3f}")
ax2.set_xlabel("Total perturbation of best boundary solution")
ax2.set_ylabel("Count (seeds)")
ax2.set_title("Perturbation cost of closest-to-boundary solution per seed", fontweight="bold")
ax2.legend(fontsize=7)

fig.tight_layout()
plt.show()

## 8 — Genotype Sparsity & Modality Balance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Sparsity distribution of Pareto solutions
ax = axes[0]
ax.hist(pareto_df["sparsity"], bins=30, color="#2176AE", alpha=0.8, edgecolor="white")
ax.axvline(pareto_df["sparsity"].median(), color="black", ls="--", lw=1.5,
          label=f"Median: {pareto_df['sparsity'].median():.3f}")
ax.set_xlabel("Genotype sparsity (fraction zero)")
ax.set_ylabel("Count")
ax.set_title("Pareto genotype sparsity", fontweight="bold")
ax.legend(fontsize=7)

# Image vs text active genes
ax = axes[1]
total_active = pareto_df["img_active"] + pareto_df["txt_active"]
img_frac = pareto_df["img_active"] / total_active.clip(lower=1)
ax.hist(img_frac, bins=30, color="#2176AE", alpha=0.8, edgecolor="white")
ax.axvline(img_frac.median(), color="black", ls="--", lw=1.5,
          label=f"Median: {img_frac.median():.3f}")
ax.set_xlabel("Image fraction of active genes")
ax.set_ylabel("Count")
ax.set_title("Modality balance (Pareto solutions)", fontweight="bold")
ax.legend(fontsize=7)

# Sparsity vs TgtBal scatter
ax = axes[2]
ax.scatter(pareto_df["sparsity"], pareto_df["TgtBal"], s=6, alpha=0.3,
          color="#2176AE", rasterized=True)
ax.set_xlabel("Sparsity")
ax.set_ylabel("TgtBal")
ax.set_title("Sparsity vs boundary distance", fontweight="bold")
ax.set_yscale("log")

fig.tight_layout()
plt.show()

## 9 — First vs Last Generation

In [ ]:
# Pooled first vs last generation fitness distributions (converged seeds only)
gen0_vals = {c: [] for c in OBJ_SHORT}
genL_vals = {c: [] for c in OBJ_SHORT}

for si in conv_idx:
    df = traces[si]
    g0 = df[df["generation"] == 0]
    gL = df[df["generation"] == df["generation"].max()]
    for col, name in zip(OBJ_COLS, OBJ_SHORT):
        gen0_vals[name].extend(g0[col].values)
        genL_vals[name].extend(gL[col].values)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, OBJ_SHORT):
    data = [gen0_vals[name], genL_vals[name]]
    parts = ax.violinplot(data, positions=[0, 1], showmeans=True, showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor("#2176AE")
        pc.set_alpha(0.6)
    for key in ["cmeans", "cmedians", "cmins", "cmaxes", "cbars"]:
        if key in parts:
            parts[key].set_color("black")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Gen 0", "Gen 49"])
    ax.set_title(name, fontweight="bold")
    if name == "TgtBal":
        ax.set_yscale("symlog", linthresh=0.01)

fig.suptitle(f"Pooled population fitness: generation 0 vs 49 (n={len(conv_idx)} converged seeds)",
            fontsize=11, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## 10 — Objective Correlations

In [ ]:
from scipy.stats import spearmanr

# Pooled Spearman correlation across all converged seeds
pooled = pd.concat([traces[i] for i in conv_idx], ignore_index=True)
n = len(OBJ_COLS)
corr = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        corr[i, j], _ = spearmanr(pooled[OBJ_COLS[i]], pooled[OBJ_COLS[j]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(OBJ_SHORT, rotation=45, ha="right")
ax.set_yticklabels(OBJ_SHORT)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center",
               fontsize=11, color="white" if abs(corr[i,j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, label="Spearman ρ", shrink=0.8)
ax.set_title(f"Pooled objective correlations (n={len(conv_idx)} seeds)", fontweight="bold")
fig.tight_layout()
plt.show()

## 11 — Stuck Seeds: Failure Analysis

In [ ]:
stuck_idx = seed_df[~seed_df["converged"]].index.values

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

# Row 1: Convergence curves for stuck seeds
ax = axes[0, 0]
for si in stuck_idx:
    ax.plot(convs[si]["generation"], convs[si]["pareto_min_TgtBal"], lw=1.5,
           label=f"S{si}: {seed_df.loc[si,'pair'][:25]}")
ax.set_xlabel("Generation"); ax.set_ylabel("min TgtBal")
ax.set_title("Stuck seed convergence", fontweight="bold")
ax.legend(fontsize=5, ncol=2)

# Row 1: Stuck seed initial vs final TgtBal
ax = axes[0, 1]
ax.scatter(seed_df.loc[stuck_idx, "initial_tgtbal"],
          seed_df.loc[stuck_idx, "final_tgtbal"],
          c="#E0607E", s=60, edgecolors="black", linewidth=0.5, zorder=5)
for si in stuck_idx:
    ax.annotate(f"S{si}", (seed_df.loc[si,"initial_tgtbal"], seed_df.loc[si,"final_tgtbal"]),
               fontsize=6, ha="left", va="bottom")
ax.plot([0, 1], [0, 1], "k:", lw=0.5)
ax.set_xlabel("Initial TgtBal"); ax.set_ylabel("Final TgtBal")
ax.set_title("Initial vs final (stuck seeds)", fontweight="bold")

# Row 1: Class-pair frequency in stuck vs converged
ax = axes[0, 2]
stuck_classes = Counter(seed_df.loc[stuck_idx, "class_a"])
all_classes = Counter(seed_df["class_a"])
stuck_rate = {c: stuck_classes.get(c, 0) / all_classes[c] for c in all_classes}
classes_sorted = sorted(stuck_rate, key=stuck_rate.get, reverse=True)
x = np.arange(len(classes_sorted))
ax.bar(x, [stuck_rate[c] for c in classes_sorted], color="#E0607E", alpha=0.8, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([c[:12] for c in classes_sorted], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Stuck fraction")
ax.set_title("Stuck rate by source class", fontweight="bold")

# Row 2: P(A) vs P(B) for stuck seeds
for col, si in enumerate(stuck_idx[:3]):
    ax = axes[1, col]
    df = traces[si]
    sc = ax.scatter(df["p_class_a"], df["p_class_b"], c=df["generation"],
                   cmap="viridis", s=4, alpha=0.4, rasterized=True)
    ax.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect("equal")
    ax.set_xlabel("P(A)"); ax.set_ylabel("P(B)")
    ax.set_title(f"S{si}: {seed_df.loc[si,'pair'][:25]}", fontsize=8, fontweight="bold")

# Row 3: Population TgtBal distribution evolution for worst stuck seed
worst_stuck = stuck_idx[np.argmax(seed_df.loc[stuck_idx, "final_tgtbal"].values)]
df = traces[worst_stuck]
gens_to_show = [0, 10, 25, 49]
for col, g in enumerate(gens_to_show[:3]):
    ax = axes[2, col]
    gen_data = df[df["generation"] == g]["fitness_TgtBal"].values
    ax.hist(gen_data, bins=15, color="#E0607E" if g == 49 else "#2176AE", alpha=0.7, edgecolor="white")
    ax.set_xlabel("TgtBal"); ax.set_ylabel("Count")
    ax.set_title(f"S{worst_stuck} Gen {g} TgtBal dist", fontweight="bold", fontsize=9)

fig.tight_layout()
plt.show()

# Summary
print(f"Stuck seeds: {len(stuck_idx)}/55")
print(f"Class distribution: {dict(stuck_classes)}")

## 12 — Spatial Perturbation Heatmaps

Mean patch activation rate across Pareto solutions, overlaid on a 32x32 VQGAN grid.
Shown for 6 representative seeds (3 best converged, 3 highest pareto count).

In [ ]:
# Pick 6 diverse seeds: 3 best-converged, 3 most pareto solutions
best_3 = seed_df[seed_df["converged"]].nsmallest(3, "final_tgtbal").index.tolist()
most_pareto = seed_df.nlargest(3, "n_pareto").index.tolist()
show_seeds = list(dict.fromkeys(best_3 + most_pareto))[:6]  # deduplicate, keep order

fig, axes = plt.subplots(2, len(show_seeds), figsize=(3.5 * len(show_seeds), 7))

for col, si in enumerate(show_seeds):
    ctx = contexts[si]
    positions = np.array(ctx["image_patch_positions"])
    img_dim = len(positions)
    genos = np.array([p["genotype"] for p in pareto_data[si]])
    img_genos = genos[:, :img_dim]
    activation = (img_genos != 0).mean(axis=0)

    grid = np.zeros((32, 32))
    for pos, act in zip(positions, activation):
        grid[pos[0], pos[1]] = act

    origin = seed_dirs[si] / "origin.png"
    if origin.exists():
        axes[0, col].imshow(Image.open(origin).resize((256, 256)))
    axes[0, col].axis("off")
    axes[0, col].set_title(f"S{si}: {stats_list[si]['class_a'][:10]}\nvs {stats_list[si]['class_b'][:10]}",
                          fontsize=7, fontweight="bold")

    im = axes[1, col].imshow(grid, cmap="hot", interpolation="nearest", vmin=0, vmax=1)
    axes[1, col].set_title(f"n_pareto={stats_list[si]['n_pareto']}", fontsize=7)

fig.colorbar(im, ax=axes[1].tolist(), label="Activation rate", shrink=0.8, pad=0.02)
fig.tight_layout()
plt.show()

## 13 — Summary Statistics

In [ ]:
print("=" * 70)
print("RUN 03 — CADENCE EXPERIMENT")
print("=" * 70)

print(f"\nSetup: Qwen3.5-9B | 50 categories | gap=1.0 | 50 gen x 30 pop")
print(f"Seeds: {N} | Evaluations: {N*50*30:,} | Runtime: {seed_df['runtime'].sum()/3600:.1f}h")

print(f"\nConvergence (TgtBal < 0.01):")
print(f"  Converged: {seed_df['converged'].sum()}/{N} ({seed_df['converged'].mean()*100:.0f}%)")
print(f"  Median convergence gen: {np.median([g for g in gen_conv if g < 51]):.0f}")
print(f"  Final TgtBal: mean={seed_df['final_tgtbal'].mean():.4f}, median={seed_df['final_tgtbal'].median():.6f}")
print(f"  Best: {seed_df['final_tgtbal'].min():.6f}")

print(f"\nPareto archive:")
print(f"  Total: {len(pareto_df):,} solutions")
print(f"  Per seed: mean={seed_df['n_pareto'].mean():.0f}, range=[{seed_df['n_pareto'].min()}, {seed_df['n_pareto'].max()}]")
print(f"  TgtBal<0.01: {(pareto_df['TgtBal']<0.01).sum()}/{len(pareto_df)} ({(pareto_df['TgtBal']<0.01).mean()*100:.0f}%)")

print(f"\nPerturbation (Pareto solutions, converged seeds):")
cp = pareto_df[pareto_df["seed"].isin(conv_idx)]
print(f"  MatDist: mean={cp['MatDist'].mean():.4f}, median={cp['MatDist'].median():.4f}")
print(f"  TextDist: mean={cp['TextDist'].mean():.4f}, median={cp['TextDist'].median():.4f}")
print(f"  Sparsity: mean={cp['sparsity'].mean():.3f}, median={cp['sparsity'].median():.3f}")

print(f"\nStuck seeds ({len(stuck_idx)}):")
for si in stuck_idx:
    print(f"  S{si:2d}: {seed_df.loc[si,'pair']:45s} TB={seed_df.loc[si,'final_tgtbal']:.4f}")

## 14 — Cross-Run Comparison

| | Run 01 (5-obj) | Run 02 (4-obj) | Run 03 (cadence) |
|---|---|---|---|
| Model | Qwen3.5-4B | Qwen3.5-4B | Qwen3.5-9B |
| Objectives | 3 + Conc + ArchiveSparsity | 3 | 3 |
| Generations x Pop | 30 x 20 | 50 x 30 | 50 x 30 |
| Categories (SUT) | 15 (multi-class) | 2 (binary) | 2 (binary) |
| Seed filter (gap) | 2.0 | 2.0 | 1.0 |
| Seeds | 3 | 5 | 55 |

In [ ]:
# ── Load comparable data from all three runs ──
from pathlib import Path

def load_run_summary(run_path):
    """Extract per-seed summary from any run directory."""
    sds = sorted(run_path.glob("vlm_boundary_seed_*"))
    rows = []
    for sd in sds:
        sp = sd / "stats.json"
        if not sp.exists():
            continue
        s = json.loads(sp.read_text())

        # TgtBal trajectory — from convergence.parquet if available, else trace
        conv_path = sd / "convergence.parquet"
        if conv_path.exists():
            conv = pd.read_parquet(conv_path)
            init_tb = conv["pareto_min_TgtBal"].iloc[0]
            final_tb = conv["pareto_min_TgtBal"].iloc[-1]
            tgtbal_curve = conv["pareto_min_TgtBal"].values
        else:
            t = pd.read_parquet(sd / "trace.parquet")
            best_pg = t.groupby("generation")["fitness_TgtBal"].min()
            init_tb = best_pg.iloc[0]
            final_tb = best_pg.iloc[-1]
            tgtbal_curve = best_pg.values

        # Best pareto TgtBal
        pfiles = sorted(sd.glob("pareto_*.json"), key=lambda p: int(p.stem.split("_")[1]))
        pareto_tbs = [json.loads(f.read_text())["fitness"][2] for f in pfiles]
        pareto_mds = [json.loads(f.read_text())["fitness"][0] for f in pfiles]
        pareto_tds = [json.loads(f.read_text())["fitness"][1] for f in pfiles]

        rows.append({
            "pair": f"{s['class_a']} vs {s['class_b']}",
            "init_tb": init_tb, "final_tb": final_tb,
            "best_pareto_tb": min(pareto_tbs),
            "median_pareto_md": np.median(pareto_mds),
            "median_pareto_td": np.median(pareto_tds),
            "n_pareto": s["n_pareto"],
            "gens": s["generations"], "pop": s["pop_size"],
            "runtime": s["runtime_seconds"],
            "tgtbal_curve": tgtbal_curve,
        })
    return pd.DataFrame(rows)

r01 = load_run_summary(Path("../runs/Archive/01_5obj-sparsityActive"))
r02 = load_run_summary(Path("../runs/02_4obj"))
r03 = load_run_summary(Path("../runs/03_cadence"))

run_meta = {
    "01: 5-obj\n(4B, 15-class)": r01,
    "02: 3-obj\n(4B, binary)": r02,
    "03: cadence\n(9B, binary)": r03,
}
run_colors = {"01: 5-obj\n(4B, 15-class)": "#E0607E",
              "02: 3-obj\n(4B, binary)": "#FBB13C",
              "03: cadence\n(9B, binary)": "#2176AE"}

print(f"Run 01: {len(r01)} seeds | Run 02: {len(r02)} seeds | Run 03: {len(r03)} seeds")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# ── Row 1, Col 1: Convergence curves (all seeds, all runs) ──
ax = axes[0, 0]
for label, df in run_meta.items():
    color = run_colors[label]
    for _, row in df.iterrows():
        curve = row["tgtbal_curve"]
        gens = np.arange(len(curve))
        ax.plot(gens, curve, color=color, lw=0.5, alpha=0.4)
    # Mean curve
    max_gen = df["tgtbal_curve"].apply(len).min()
    stacked = np.array([c[:max_gen] for c in df["tgtbal_curve"]])
    ax.plot(np.arange(max_gen), stacked.mean(axis=0), color=color, lw=2.5,
           label=label.replace("\n", " "))
ax.set_yscale("log")
ax.set_xlabel("Generation")
ax.set_ylabel("min TgtBal (log)")
ax.set_title("Convergence trajectories", fontweight="bold")
ax.legend(fontsize=6)

# ── Row 1, Col 2: Final TgtBal distribution ──
ax = axes[0, 1]
positions = []
data = []
colors = []
labels = []
for i, (label, df) in enumerate(run_meta.items()):
    data.append(df["final_tb"].values)
    positions.append(i)
    colors.append(run_colors[label])
    labels.append(label)

bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("Final TgtBal")
ax.set_title("Final TgtBal distribution", fontweight="bold")
ax.set_yscale("symlog", linthresh=0.001)

# ── Row 1, Col 3: Convergence rate ──
ax = axes[0, 2]
conv_rates = []
n_seeds = []
for label, df in run_meta.items():
    cr = (df["final_tb"] < 0.01).mean()
    conv_rates.append(cr)
    n_seeds.append(len(df))
bars = ax.bar(range(len(run_meta)), conv_rates, color=list(run_colors.values()), alpha=0.8, edgecolor="white")
for i, (cr, ns) in enumerate(zip(conv_rates, n_seeds)):
    ax.text(i, cr + 0.02, f"{cr*100:.0f}%\n(n={ns})", ha="center", fontsize=8)
ax.set_xticks(range(len(run_meta)))
ax.set_xticklabels(list(run_meta.keys()), fontsize=7)
ax.set_ylabel("Convergence rate (TgtBal < 0.01)")
ax.set_title("Convergence rate", fontweight="bold")
ax.set_ylim(0, 1.15)

# ── Row 2, Col 1: Pareto archive size per seed ──
ax = axes[1, 0]
for i, (label, df) in enumerate(run_meta.items()):
    ax.scatter([i] * len(df), df["n_pareto"], color=run_colors[label], s=30, alpha=0.6, edgecolors="white")
    ax.scatter(i, df["n_pareto"].median(), color="black", s=80, marker="_", zorder=5)
ax.set_xticks(range(len(run_meta)))
ax.set_xticklabels(list(run_meta.keys()), fontsize=7)
ax.set_ylabel("Pareto solutions per seed")
ax.set_title("Pareto archive size", fontweight="bold")

# ── Row 2, Col 2: Perturbation cost (median MatDist of pareto solutions) ──
ax = axes[1, 1]
for i, (label, df) in enumerate(run_meta.items()):
    ax.scatter([i] * len(df), df["median_pareto_md"], color=run_colors[label], s=30, alpha=0.6, edgecolors="white")
    ax.scatter(i, df["median_pareto_md"].median(), color="black", s=80, marker="_", zorder=5)
ax.set_xticks(range(len(run_meta)))
ax.set_xticklabels(list(run_meta.keys()), fontsize=7)
ax.set_ylabel("Median Pareto MatDist")
ax.set_title("Image perturbation cost", fontweight="bold")

# ── Row 2, Col 3: Runtime per seed ──
ax = axes[1, 2]
for i, (label, df) in enumerate(run_meta.items()):
    ax.scatter([i] * len(df), df["runtime"] / 60, color=run_colors[label], s=30, alpha=0.6, edgecolors="white")
    ax.scatter(i, df["runtime"].median() / 60, color="black", s=80, marker="_", zorder=5)
ax.set_xticks(range(len(run_meta)))
ax.set_xticklabels(list(run_meta.keys()), fontsize=7)
ax.set_ylabel("Runtime per seed (min)")
ax.set_title("Computational cost", fontweight="bold")

fig.suptitle("Cross-Run Comparison", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Tabular summary
print(f"{'':30s} {'Run 01 (5-obj)':>18s} {'Run 02 (3-obj)':>18s} {'Run 03 (cadence)':>18s}")
print("-" * 86)

metrics = [
    ("Seeds", [len(r01), len(r02), len(r03)]),
    ("Generations x Pop", [f"{r01['gens'].iloc[0]}x{r01['pop'].iloc[0]}",
                           f"{r02['gens'].iloc[0]}x{r02['pop'].iloc[0]}",
                           f"{r03['gens'].iloc[0]}x{r03['pop'].iloc[0]}"]),
    ("Converged (TB<0.01)", [f"{(r01['final_tb']<0.01).sum()}/{len(r01)} ({(r01['final_tb']<0.01).mean()*100:.0f}%)",
                             f"{(r02['final_tb']<0.01).sum()}/{len(r02)} ({(r02['final_tb']<0.01).mean()*100:.0f}%)",
                             f"{(r03['final_tb']<0.01).sum()}/{len(r03)} ({(r03['final_tb']<0.01).mean()*100:.0f}%)"]),
    ("Final TB mean", [f"{r01['final_tb'].mean():.4f}", f"{r02['final_tb'].mean():.4f}", f"{r03['final_tb'].mean():.4f}"]),
    ("Final TB median", [f"{r01['final_tb'].median():.4f}", f"{r02['final_tb'].median():.4f}", f"{r03['final_tb'].median():.6f}"]),
    ("Best TB", [f"{r01['final_tb'].min():.4f}", f"{r02['final_tb'].min():.6f}", f"{r03['final_tb'].min():.6f}"]),
    ("Pareto/seed (median)", [f"{r01['n_pareto'].median():.0f}", f"{r02['n_pareto'].median():.0f}", f"{r03['n_pareto'].median():.0f}"]),
    ("Total Pareto", [f"{r01['n_pareto'].sum()}", f"{r02['n_pareto'].sum()}", f"{r03['n_pareto'].sum()}"]),
    ("MatDist (median/seed)", [f"{r01['median_pareto_md'].median():.4f}",
                               f"{r02['median_pareto_md'].median():.4f}",
                               f"{r03['median_pareto_md'].median():.4f}"]),
    ("Runtime/seed (min)", [f"{r01['runtime'].median()/60:.1f}", f"{r02['runtime'].median()/60:.1f}", f"{r03['runtime'].median()/60:.1f}"]),
]

for name, vals in metrics:
    print(f"{name:30s} {str(vals[0]):>18s} {str(vals[1]):>18s} {str(vals[2]):>18s}")